# Day 3: Taxi Data Cleaning

**Goal:** clean the taxi data, add the pickup service zone from the official NYC zone lookup, and calculate the share of revenue from airport pickups.

This notebook keeps questionable but potentially valid rides (zero distance and zero passengers) and documents the decision. It removes rides with missing or invalid pickup/drop-off times, fills missing category/location values with `Unknown`, and uses a left merge so rides are not silently dropped during the zone lookup.


## 1. Load the taxi rides and zone lookup

In [9]:
import pandas as pd

taxi_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

rides = pd.read_csv(taxi_url)
zones = pd.read_csv(zone_url)

print("Taxi rows loaded:", len(rides))
print("Zone lookup rows loaded:", len(zones))
display(rides.head())
display(zones.head())


Taxi rows loaded: 6433
Zone lookup rows loaded: 265


,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## 2. Check the starting data

In [10]:
print("Missing values in taxi data:")
display(rides.isna().sum().to_frame("missing_rows"))

cleaning_log = [
    f"Started with {len(rides)} taxi rows."
]


Missing values in taxi data:


,missing_rows
pickup,0
dropoff,0
passengers,0
distance,0
fare,0
tip,0
tolls,0
total,0
color,0
payment,44


## 3. Remove rides with missing or invalid times

In [11]:
# Convert timestamps to datetime values so we can compare them safely.
rides["pickup"] = pd.to_datetime(rides["pickup"], errors="coerce")
rides["dropoff"] = pd.to_datetime(rides["dropoff"], errors="coerce")

# A ride is invalid if either time is missing, or drop-off is not after pickup.
bad_time = (
    rides["pickup"].isna()
    | rides["dropoff"].isna()
    | (rides["dropoff"] <= rides["pickup"])
)
bad_time_count = int(bad_time.sum())
rides = rides.loc[~bad_time].copy()

cleaning_log.append(
    f"Removed {bad_time_count} rides with missing timestamps or drop-off time not after pickup; these rides cannot provide a reliable trip duration."
)
print("Rows after time check:", len(rides))


Rows after time check: 6427


## 4. Handle missing payment types

In [12]:
missing_payment = int(rides["payment"].isna().sum())
rides["payment"] = rides["payment"].fillna("Unknown")

cleaning_log.append(
    f"Filled {missing_payment} missing payment values with 'Unknown' rather than guessing how the ride was paid."
)
print("Missing payment values remaining:", int(rides["payment"].isna().sum()))


Missing payment values remaining: 0


## 5. Handle missing pickup/drop-off zones and boroughs

In [13]:
location_columns = [
    "pickup_zone",
    "dropoff_zone",
    "pickup_borough",
    "dropoff_borough",
]

for column in location_columns:
    missing_count = int(rides[column].isna().sum())
    rides[column] = rides[column].fillna("Unknown")
    cleaning_log.append(
        f"Filled {missing_count} missing values in {column} with 'Unknown'; the true location cannot be safely inferred from this table alone."
    )
    print(f"{column}: filled {missing_count} missing values")


pickup_zone: filled 22 missing values
dropoff_zone: filled 39 missing values
pickup_borough: filled 22 missing values
dropoff_borough: filled 39 missing values


## 6. Check zero distance and zero passengers

In [14]:
zero_distance = int((rides["distance"] == 0).sum())
zero_passengers = int((rides["passengers"] == 0).sum())

# Keep these rows for now: they may be unusual, but the values alone do not
# prove the records are incorrect. We record the decision for transparency.
cleaning_log.append(
    f"Kept {zero_distance} rides with zero distance; a zero value may represent a very short trip or a data issue, so it should be investigated rather than automatically deleted."
)
cleaning_log.append(
    f"Kept {zero_passengers} rides with zero passengers; the value may be a recording issue, but there is not enough evidence here to replace or delete it."
)

print("Rides with zero distance (kept):", zero_distance)
print("Rides with zero passengers (kept):", zero_passengers)


Rides with zero distance (kept): 45
Rides with zero passengers (kept): 96


## 7. Check zone lookup uniqueness before merging

In [15]:
# Check for duplicate Borough + Zone pairs
duplicate_pairs = zones[
    zones.duplicated(["Borough", "Zone"], keep=False)
]

print("Duplicate lookup rows:", len(duplicate_pairs))

if not duplicate_pairs.empty:
    display(duplicate_pairs.sort_values(["Borough", "Zone"]))

# Keep one lookup row per Borough + Zone pair
zones = zones.drop_duplicates(
    subset=["Borough", "Zone"],
    keep="first"
)

print("Lookup rows after removing duplicates:", len(zones))
print(
    "Duplicate pairs remaining:",
    zones.duplicated(["Borough", "Zone"]).sum()
)

Duplicate lookup rows: 5


,LocationID,Borough,Zone,service_zone
102,103,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
103,104,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
104,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
55,56,Queens,Corona,Boro Zone
56,57,Queens,Corona,Boro Zone


Lookup rows after removing duplicates: 262
Duplicate pairs remaining: 0


## 8. Merge pickup service zone

In [16]:
rows_before_merge = len(rides)

pickup_lookup = zones[["Borough", "Zone", "service_zone"]].copy()

rides = rides.merge(
    pickup_lookup,
    how="left",
    left_on=["pickup_borough", "pickup_zone"],
    right_on=["Borough", "Zone"],
    validate="many_to_one",
    indicator=True,
)

rows_after_merge = len(rides)
unmatched_pickups = int((rides["_merge"] == "left_only").sum())

cleaning_log.append(
    f"Merged pickup service zone using a left join on borough and zone: {rows_before_merge} rows before and {rows_after_merge} rows after. A left join preserves all ride rows; the row count should stay the same."
)
cleaning_log.append(
    f"Found {unmatched_pickups} pickup rows with no matching lookup entry; unmatched values are kept and recorded as Unknown rather than dropping rides."
)

print("Rows before merge:", rows_before_merge)
print("Rows after merge:", rows_after_merge)
print("Unmatched pickup locations:", unmatched_pickups)

rides = rides.drop(columns=["Borough", "Zone", "_merge"])
missing_service_zone = int(rides["service_zone"].isna().sum())
rides["service_zone"] = rides["service_zone"].fillna("Unknown")
cleaning_log.append(
    f"Filled {missing_service_zone} missing pickup service_zone values with 'Unknown' because no matching service-zone value was available."
)
display(rides[["pickup_borough", "pickup_zone", "service_zone"]].head())


Rows before merge: 6427
Rows after merge: 6427
Unmatched pickup locations: 22


,pickup_borough,pickup_zone,service_zone
0,Manhattan,Lenox Hill West,Yellow Zone
1,Manhattan,Upper West Side South,Yellow Zone
2,Manhattan,Alphabet City,Yellow Zone
3,Manhattan,Hudson Sq,Yellow Zone
4,Manhattan,Midtown East,Yellow Zone


## 9. Save the cleaned dataset

In [17]:
# Save beside this notebook. In VS Code, this should be inside the week1 folder.
rides.to_csv("cleaned.csv", index=False)
print("Saved cleaned.csv with", len(rides), "rows.")


Saved cleaned.csv with 6427 rows.


## 10. What share of revenue came from airport pickups?

In [18]:
total_revenue = rides["total"].sum()
airport_revenue = rides.loc[rides["service_zone"] == "Airports", "total"].sum()

if total_revenue != 0:
    airport_share = (airport_revenue / total_revenue) * 100
else:
    airport_share = 0

print(f"Total revenue in cleaned data: ${total_revenue:,.2f}")
print(f"Revenue from airport pickups: ${airport_revenue:,.2f}")
print(f"Share of total revenue from airport pickups: {airport_share:.2f}%")
print(
    f"Plain-English answer: Airport pickups contributed {airport_share:.2f}% "
    "of the total revenue in the cleaned taxi data."
)


Total revenue in cleaned data: $119,019.87
Revenue from airport pickups: $14,624.24
Share of total revenue from airport pickups: 12.29%
Plain-English answer: Airport pickups contributed 12.29% of the total revenue in the cleaned taxi data.


## 11. Final checks and write the cleaning log

In [19]:
print("Final rows:", len(rides))
print("Exact duplicate rows:", int(rides.duplicated().sum()))
print("Pickup service-zone counts:")
display(rides["service_zone"].value_counts(dropna=False).to_frame("rides"))

cleaning_log.extend([
    "So what: filling missing labels with 'Unknown' keeps rides available for analysis without pretending to know missing details.",
    "So what: the left merge preserves the ride records; unmatched pickup locations are visible instead of silently disappearing.",
    "So what: airport revenue share is calculated from the cleaned data and the added pickup service-zone label."
])

with open("cleaning_log.md", "w", encoding="utf-8") as file:
    for line in cleaning_log:
        file.write("- " + line + "\n")

print("Saved cleaning_log.md")
print("\n".join("- " + line for line in cleaning_log))


Final rows: 6427
Exact duplicate rows: 0
Pickup service-zone counts:


,rides
service_zone,
Yellow Zone,4854
Boro Zone,1254
Airports,297
Unknown,22


Saved cleaning_log.md
- Started with 6433 taxi rows.
- Removed 6 rides with missing timestamps or drop-off time not after pickup; these rides cannot provide a reliable trip duration.
- Filled 43 missing payment values with 'Unknown' rather than guessing how the ride was paid.
- Filled 22 missing values in pickup_zone with 'Unknown'; the true location cannot be safely inferred from this table alone.
- Filled 39 missing values in dropoff_zone with 'Unknown'; the true location cannot be safely inferred from this table alone.
- Filled 22 missing values in pickup_borough with 'Unknown'; the true location cannot be safely inferred from this table alone.
- Filled 39 missing values in dropoff_borough with 'Unknown'; the true location cannot be safely inferred from this table alone.
- Kept 45 rides with zero distance; a zero value may represent a very short trip or a data issue, so it should be investigated rather than automatically deleted.
- Kept 96 rides with zero passengers; the value may b

In [20]:
from pathlib import Path

# Save the cleaned data in the week1 folder
output_folder = Path.cwd()

rides.to_csv(output_folder / "cleaned.csv", index=False)

# Save the cleaning decisions
with open(output_folder / "cleaning_log.md", "w", encoding="utf-8") as file:
    file.write("# Day 3 Cleaning Log\n\n")
    for decision in cleaning_log:
        file.write(f"- {decision}\n")

print("Cleaned CSV saved to:", output_folder / "cleaned.csv")
print("Cleaning log saved to:", output_folder / "cleaning_log.md")
print("Rows saved:", len(rides))

Cleaned CSV saved to: f:\web3geeks-ai-bootcamp\week1\cleaned.csv
Cleaning log saved to: f:\web3geeks-ai-bootcamp\week1\cleaning_log.md
Rows saved: 6427
